### Grades Predictions
The code connectes to the server using pyodbc, reads file to make predictions on, retrieves 'pickled' models and makes predictions.  The resulting file is writted out in csv format.

In [ ]:
# use only for Jupyter notebooks, not for py. version
%reset -f               

import pandas as pd
import numpy as np
import pyodbc
import sqlalchemy as sa
from dotenv import load_dotenv
import os


In [2]:
import warnings
warnings.filterwarnings('ignore')

#### Description of the Variables and Structure of the Scoring System
##### Structure of Scoring Equations and Series of Variables comprazing the Scoring Models:

There are three pieces to a catalog scoring equation.<br>
1.	Probability of a Customer Purchasing, Next Year (30%).<br>
2.	Amount a Customer Will Spend if Customer Purchases, Next Year ($100.00).<br>
3.	Percentage of Next Year’s Spend That is Organic (Non-Catalog) (60%).<br>

The catalog score is derived as follows:<br>
•	(Probability of Purchase) * (Predicted Spend) * (1 – Predicted Organic Percentage).<br>


A series of variables comprise the scoring models.<br>
•	ROOT_REC = Square root of months since last purchase.<br>
•	HST_FREQ = Number of life-to-date purchases.<br>
•	HST_DEMD = Life-to-Date Demand spent.<br>
•	HST_AOV = (HST_DEMD / HST_FREQ).<br>
•	HST_AOV000 = HST_AOV / 1000.<br>
•	Merchandise Indicators … 1 = Ever Bought, 0 = Never Bought. Derived from the High Level Category Variable Sent to Me.<br>
o	MR00 = All Other Merchandise<br>
o	MR01 = Adjustments<br>
o	MR02 = Bibles<br>
o	MR03 = Books<br>
o	MR04 = Church Supplies<br>
o	MR05 = Closeouts<br>
o	MR06 = Damaged<br>
o	MR07 = Downloads<br>
o	MR08 = Exclusives<br>
o	MR09 = Gifts<br>
o	MR10 = HomeSchool<br>
o	MR11 = Music<br>
o	MR12 = Videos<br>
•	Channel Indicators … 1 = Ever Bought, 0 = Never Bought. Derived from each attributed channel sent to me.<br>
o	HST_AFFI (affiliates)<br>
o	HST_CATG (catalog)<br>
o	HST_CATI (catalog insert)<br>
o	HST_EMAI (email)<br>
o	HST_SRCO (organic search)<br>
o	HST_OTHR (all other)<br>
o	HST_PBRN (paid search brand)<br>
o	HST_PNON (paid search non-branded)<br>
o	HST_SHOP (shopping / PLA).<br>
o	HST_TEXT (sms)<br>
o	HST_SOCI (social)<br>
•	HST_SHIP (sum all paid shipping, then divide by historical demand spent).<br>
•	HST_ORGN (sum all demand spent except for catalog attribution and catalog insert attribution, divide by sum of all demand spent).<br>
•	HST_CLICK = Sum of all email clicks, past year.<br>
•	HST_VISI = Sum of all website visits, past year.<br><br><br>
•	ROOT_ECL = Square Root of HST_CLCK.   !!!!Edit! Edit!!!!! Edit!!!!! Currently, EOOT_ECL<br>    <br> <br>
•	ROOT_VIS = Square Root of HST_VISI.<br>



#### I. Establish Connection to the Server and read in Input file to make prediction

##### Connection with hard coded user_id and password (the definitions are above):

In [ ]:
# Connection string for Non-trusted connection: credentials are hard coded above
load_dotenv()  # loads .env from current directory or parent directories
mssqlDriver = os.getenv("DB_DRIVER")
mssqlServer = os.getenv("DB_SERVER")
mssqlUser = os.getenv("DB_USERNAME")
mssqlPw = os.getenv("DB_PASSWORD")
mssqlClient = os.getenv("B_DATABASE")

conn_str = f'DRIVER={mssqlDriver};SERVER={mssqlServer};DATABASE={mssqlClient}_CDM;TRUSTED_CONNECTION=no;UID={mssqlUser};PWD={mssqlPw};'

# connect to the SQL Server
conx = pyodbc.connect(conn_str)

In [ ]:
# Select the table to read the data:
query = "select * from F_Customer_Model"

In [9]:
with pyodbc.connect(conn_str) as conx:                   # no need to close the connection (conx.close())
    cursor = conx.cursor()                               # create cursor
    cursor.execute(query)                                # execute query
    # or, execute query with parameter:
    # cursor.execute(query, 'Active')
    data = cursor.fetchall()                             # fetch the result

In [10]:
pd.set_option('display.max_columns', None)

In [ ]:
# create column list for Model input:
# (use Source_Customer_ID as 'customerID', which is primary identifyer'
column_list = ['Customer_Dim_Id','customerID', 'ROOT_REC', 'HST_FREQ', 'HST_DEMD', 'HST_AOV',
       'MR00', 'MR01', 'MR02', 'MR03', 'MR04', 'MR05', 'MR06',
       'MR07', 'MR08', 'MR09', 'MR10', 'MR11', 'MR12', 'HST_AFFI', 'HST_CATG',
       'HST_CATI', 'HST_EMAI', 'HST_SRCO', 'HST_OTHR', 'HST_PBRN', 'HST_PNON',
       'HST_SHOP', 'HST_TEXT', 'HST_SOCI', 'HST_SHIP', 'HST_ORGN', 'HST_CLICK',
       'HST_VISI', 'ROOT_ECL', 'ROOT_VIS', 'UPDATE_DT','POLLING_ID', 'HST_AOV000']
      
    

In [12]:
# define column list for our dataframe, then generate dataframe
#column_list = ['Customer_DIM_ID','Recency_Group','Lifecycle_Group']
df = pd.DataFrame((tuple(t) for t in data), columns = column_list)
df.head()

,Customer_Dim_Id,customerID,ROOT_REC,HST_FREQ,HST_DEMD,HST_AOV,MR00,MR01,MR02,MR03,MR04,MR05,MR06,MR07,MR08,MR09,MR10,MR11,MR12,HST_AFFI,HST_CATG,HST_CATI,HST_EMAI,HST_SRCO,HST_OTHR,HST_PBRN,HST_PNON,HST_SHOP,HST_TEXT,HST_SOCI,HST_SHIP,HST_ORGN,HST_CLICK,HST_VISI,ROOT_ECL,ROOT_VIS,UPDATE_DT,POLLING_ID,HST_AOV000
0,5485676,6709175,6.0,1,62.98,62.98,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,1.0,0,0,0.0,0.0,2024-04-28 13:34:16.907,509,0.06298
1,5485678,5579623,6.0,1,11.98,11.98,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.750417,1.0,0,0,0.0,0.0,2024-04-28 13:34:16.907,509,0.01198
2,5485701,2812261,6.0,1,61.99,61.99,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.000000,1.0,0,0,0.0,0.0,2024-04-28 13:34:16.907,509,0.06199
3,5485703,1987891,6.0,1,27.48,27.48,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0.217977,1.0,0,0,0.0,0.0,2024-04-28 13:34:16.907,509,0.02748
4,5485710,6709181,6.0,1,60.98,60.98,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0.104296,1.0,0,0,0.0,0.0,2024-04-28 13:34:16.907,509,0.06098


In [13]:
df.shape

(7639169, 39)

#### II. Retrieve 'pickled' Models, Make Predictions and Calculate Grades
Load saved models; make predictions and calculate customer grades. <br> 
NOTE: Recreate the list of explanatory and response variables if using as separate module

In [14]:
import statsmodels.api as sm

In [15]:
df_prediction = df.copy()
df_prediction = df.fillna(0)

##### 2.1 Define lists of explanatory and response variables for each model; retrieve 'pickled' models and make predictions

In [ ]:
# Define lists of explanatory variables for each model
col_predr = ['ROOT_REC','HST_FREQ','HST_AOV000','MR00','MR01','MR03', 'MR04', 'MR05','MR06','MR07','MR08', 'MR09', 'MR10', 'MR11', 'MR12',
            'HST_AFFI', 'HST_CATG', 'HST_CATI', 'HST_EMAI', 'HST_SRCO', 'HST_OTHR','HST_PBRN','HST_PNON', 'HST_SHOP', 'HST_TEXT',
            'ROOT_ECL', 'ROOT_VIS', 'HST_ORGN']
col_preds = ['HST_DEMD', 'ROOT_VIS', 'ROOT_REC', 'ROOT_ECL']
col_predo = ['ROOT_REC','HST_FREQ','HST_ORGN','MR01','MR03','MR07','MR08','MR09', 'MR10','MR11','MR12','HST_AFFI','HST_CATG','HST_EMAI',
            'HST_SRCO','HST_OTHR', 'HST_PBRN', 'HST_PNON', 'HST_SHOP', 'HST_TEXT','HST_SOCI','ROOT_ECL','ROOT_VIS']
col_pred_ship = ['ROOT_REC','HST_FREQ','MR02','MR04','MR05','MR06','MR08','MR10','MR12','HST_AFFI','HST_EMAI','HST_PBRN','HST_PNON',
            'HST_SHOP','HST_TEXT','HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']
col_pred_affi = ['HST_FREQ','MR01','MR03','MR05','MR06','MR07','MR09','MR10','MR11','HST_AFFI','HST_CATG','HST_CATI','HST_EMAI','HST_OTHR','HST_PBRN',
            'HST_PNON','HST_SHOP','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_emai = ['ROOT_REC','HST_FREQ','MR01','MR02','MR03','MR05','MR06','MR07','MR08','MR09','MR10','MR11','MR12', 'HST_AFFI','HST_CATG',
                 'HST_CATI','HST_EMAI','HST_SRCO','HST_OTHR','HST_PNON','HST_SHOP','HST_TEXT','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_srco = ['ROOT_REC', 'MR01', 'MR03', 'MR04', 'MR07', 'MR08', 'MR09','MR10','HST_AFFI','HST_CATG','HST_CATI','HST_SRCO','HST_PBRN',
            'HST_PNON', 'HST_SHOP','HST_TEXT', 'HST_SOCI', 'ROOT_ECL', 'ROOT_VIS']

col_pred_pbrn = ['ROOT_REC','HST_FREQ','MR01','MR03','MR04', 'MR05', 'MR06', 'MR07','MR08', 'MR10', 'MR11', 'MR12', 'HST_CATG', 'HST_EMAI',
            'HST_SRCO', 'HST_PBRN','HST_PNON', 'HST_SHOP', 'ROOT_ECL', 'ROOT_VIS', 'HST_SHIP']

col_pred_pnon = ['ROOT_REC', 'HST_FREQ', 'MR01','MR03','MR04','MR05', 'MR06', 'MR08', 'MR09', 'MR10', 'HST_AFFI', 'HST_CATG', 'HST_EMAI',
            'HST_SRCO', 'HST_OTHR', 'HST_PBRN','HST_PNON', 'HST_SHOP', 'ROOT_ECL', 'ROOT_VIS', 'HST_SHIP']

col_pred_shop = ['ROOT_REC', 'HST_FREQ', 'MR01','MR02','MR05','MR06', 'MR07', 'MR08', 'MR09', 'MR10', 'MR11', 'HST_AFFI', 'HST_CATG','HST_CATI',
            'HST_EMAI','HST_SRCO','HST_OTHR','HST_PBRN','HST_PNON','HST_SHOP','HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_text = ['ROOT_REC', 'HST_FREQ', 'MR00','MR01','MR02','MR05', 'MR06', 'MR08', 'MR09', 'MR10', 'HST_AFFI','HST_CATG','HST_EMAI','HST_SRCO','HST_PBRN',
            'HST_SHOP','HST_TEXT','HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_soci = ['HST_FREQ', 'MR01','MR05','MR07','MR08', 'MR09', 'MR10', 'MR11', 'HST_CATG', 'HST_CATI','HST_SRCO','HST_OTHR','HST_PNON',
            'HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_list = [col_predr, col_preds, col_predo, col_pred_ship, col_pred_affi, col_pred_emai, col_pred_srco, col_pred_pbrn, col_pred_pnon, col_pred_shop, col_pred_text, col_pred_soci]

In [17]:
# Define list of response variables for each model
prediction_list = ['predr', 'preds', 'predo', 'pred_ship', 'pred_affi','pred_emai', 'pred_srco', 'pred_pbrn', 'pred_pnon', 'pred_shop', 'pred_text', 'pred_soci']

In [18]:
# List with names of 'pickled' models
name_list = ['predr.pickle', 'preds.pickle','predo.pickle', 'pred_ship.pickle', 'pred_affi.pickle', 'pred_emai.pickle','pred_srco.pickle', 
             'pred_pbrn.pickle','pred_pnon.pickle', 'pred_shop.pickle', 'pred_text.pickle', 'pred_soci.pickle']

In [19]:
# Retrieve 'pickled' models:
model_list = []
for name in name_list:
    model = sm.load(name)
    model_list.append(model)
  

In [20]:
# Define function to make predictions
def predictions (df, model_list, col_list, prediction_list):
    for model, col, pred_name in zip(model_list, col_list, prediction_list):
        X_pred = df[col]
        X_pred = sm.add_constant(X_pred)
        y = model.predict(X_pred)
        if pred_name == 'preds':
            df[pred_name] = y
        else:
            df[pred_name] = np.exp(y) / (1 + np.exp(y))

    return df
        

In [21]:
# Call the function:
df_result = predictions(df_prediction, model_list, col_list, prediction_list)

In [22]:
# Add Brand Value (predv) and Catalog Value (predc) – Kevin Attribute Catalog Value = (predk) if needed.   
df_result['predv'] = df_result['predr'] * df_result['preds']
df_result['predc'] = df_result['predv'] * (1 - df_result['predo'])
#df['predk'] = df['predv'] * (1 - df['predx'])


##### 2.2 Use predictions to calculate Grades

In [23]:
df = df_result.copy()

In [24]:
# Compute Grade_Brand
df['Grade_Brand'] = 'F'
df.loc[df['predv'] > 07.5906, 'Grade_Brand'] = 'D'
df.loc[df['predv'] > 15.8716, 'Grade_Brand'] = 'C'
df.loc[df['predv'] > 33.3596, 'Grade_Brand'] = 'B'
df.loc[df['predv'] > 71.8546, 'Grade_Brand'] = 'A'

In [25]:
# Compute Grade_Ship
df['Grade_Ship'] = 'F'
df.loc[df['pred_ship'] > 0.1387, 'Grade_Ship'] = 'D'
df.loc[df['pred_ship'] > 0.1700, 'Grade_Ship'] = 'C'
df.loc[df['pred_ship'] > 0.1949, 'Grade_Ship'] = 'B'
df.loc[df['pred_ship'] > 0.2358, 'Grade_Ship'] = 'A'

In [26]:
# Compute Grade_Affiliate
df['Grade_Affiliate'] = 'F'
df.loc[df['pred_affi'] > 0.0423, 'Grade_Affiliate'] = 'D'
df.loc[df['pred_affi'] > 0.0465, 'Grade_Affiliate'] = 'C'
df.loc[df['pred_affi'] > 0.0533, 'Grade_Affiliate'] = 'B'
df.loc[df['pred_affi'] > 0.1240, 'Grade_Affiliate'] = 'A'

In [27]:
# Compute Grade_Email
df['Grade_Email'] = 'F'
df.loc[df['pred_emai'] > 0.1324, 'Grade_Email'] = 'D'
df.loc[df['pred_emai'] > 0.2031, 'Grade_Email'] = 'C'
df.loc[df['pred_emai'] > 0.3324, 'Grade_Email'] = 'B'
df.loc[df['pred_emai'] > 0.5941, 'Grade_Email'] = 'A'

In [28]:
# Compute Grade_OrganicSearch
df['Grade_OrganicSearch'] = 'F'
df.loc[df['pred_srco'] > 0.1049, 'Grade_OrganicSearch'] = 'D'
df.loc[df['pred_srco'] > 0.1193, 'Grade_OrganicSearch'] = 'C'
df.loc[df['pred_srco'] > 0.2087, 'Grade_OrganicSearch'] = 'B'
df.loc[df['pred_srco'] > 0.2551, 'Grade_OrganicSearch'] = 'A'

In [29]:
# Compute Grade_PaidBrand
df['Grade_PaidBrand'] = 'F'
df.loc[df['pred_pbrn'] > 0.0164, 'Grade_PaidBrand'] = 'D'
df.loc[df['pred_pbrn'] > 0.0316, 'Grade_PaidBrand'] = 'C'
df.loc[df['pred_pbrn'] > 0.0684, 'Grade_PaidBrand'] = 'B'
df.loc[df['pred_pbrn'] > 0.1723, 'Grade_PaidBrand'] = 'A'

In [30]:
# Compute Grade_PaidNonBrand
df['Grade_PaidNonBrand'] = 'F'
df.loc[df['pred_pnon'] > 0.1113, 'Grade_PaidNonBrand'] = 'D'
df.loc[df['pred_pnon'] > 0.1348, 'Grade_PaidNonBrand'] = 'C'
df.loc[df['pred_pnon'] > 0.1622, 'Grade_PaidNonBrand'] = 'B'
df.loc[df['pred_pnon'] > 0.2270, 'Grade_PaidNonBrand'] = 'A'

In [31]:
# Compute Grade_Shopping
df['Grade_Shopping'] = 'F'
df.loc[df['pred_shop'] > 0.2500, 'Grade_Shopping'] = 'D'
df.loc[df['pred_shop'] > 0.3645, 'Grade_Shopping'] = 'C'
df.loc[df['pred_shop'] > 0.4529, 'Grade_Shopping'] = 'B'
df.loc[df['pred_shop'] > 0.5014, 'Grade_Shopping'] = 'A'

In [32]:
# Compute Grade_Text
df['Grade_Text'] = 'F'
df.loc[df['pred_text'] > 0.0291, 'Grade_Text'] = 'D'
df.loc[df['pred_text'] > 0.0351, 'Grade_Text'] = 'C'
df.loc[df['pred_text'] > 0.0445, 'Grade_Text'] = 'B'
df.loc[df['pred_text'] > 0.0646, 'Grade_Text'] = 'A'

In [33]:
# Compute Grade_Social
df['Grade_Social'] = 'F'
df.loc[df['pred_soci'] > 0.0336, 'Grade_Social'] = 'D'
df.loc[df['pred_soci'] > 0.0402, 'Grade_Social'] = 'C'
df.loc[df['pred_soci'] > 0.0521, 'Grade_Social'] = 'B'
df.loc[df['pred_soci'] > 0.2287, 'Grade_Social'] = 'A'

In [34]:
# Compute Grade_Catalog
df['Grade_Catalog'] = 'F'
df.loc[df['predc'] >= 9.0, 'Grade_Catalog'] = 'D'
df.loc[df['predc'] >= 16.0, 'Grade_Catalog'] = 'C'
df.loc[df['predc'] >= 25.0, 'Grade_Catalog'] = 'B'
df.loc[df['predc'] >= 31.0, 'Grade_Catalog'] = 'A'

In [35]:
# Compute Grade_Catavg
df['Grade_Catavg'] = 'F'
df.loc[df['predc'] >= 0.9, 'Grade_Catavg'] = 'D'
df.loc[df['predc'] >= 5.0, 'Grade_Catavg'] = 'C'
df.loc[df['predc'] >= 20.0, 'Grade_Catavg'] = 'B'
df.loc[df['predc'] >= 31.0, 'Grade_Catavg'] = 'A'

In [36]:
# Compute Grade_CatKev
#df['Grade_CatKev'] = 'F'
#df.loc[df['predk'] >= 07.0000, 'Grade_CatKev'] = 'D'
#df.loc[df['predk'] >= 13.9000, 'Grade_CatKev'] = 'C'
#df.loc[df['predk'] >= 25.0000, 'Grade_CatKev'] = 'B'
#df.loc[df['predk'] >= 31.0000, 'Grade_CatKev'] = 'A'

##### 2.3 Keep only source cusotmer ID, predictions and grades

In [37]:
columns = ['customerID', 'predr', 'preds', 'predo', 'pred_ship', 'pred_affi', 'pred_emai', 'pred_srco', 'pred_pbrn', 'pred_pnon',
           'pred_shop', 'pred_text', 'pred_soci', 'predv', 'predc',
           'Grade_Brand','Grade_Ship','Grade_Affiliate','Grade_Email','Grade_OrganicSearch',
          'Grade_PaidBrand','Grade_PaidNonBrand','Grade_Shopping','Grade_Text','Grade_Social','Grade_Catalog','Grade_Catavg']
         

In [38]:
# Add predv and predc to previously defined prediction_list; round all predictions to 4 decimal points
df_grades = df[columns]
cols_to_round = prediction_list
cols_to_round.append('predv')
cols_to_round.append('predc')
df_grades[cols_to_round] = df_grades[cols_to_round].round(decimals = 4)


In [39]:
# Add current date
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
df_grades['Date'] = date

In [40]:
df_grades.head()

,customerID,predr,preds,predo,pred_ship,pred_affi,pred_emai,pred_srco,pred_pbrn,pred_pnon,pred_shop,pred_text,pred_soci,predv,predc,Grade_Brand,Grade_Ship,Grade_Affiliate,Grade_Email,Grade_OrganicSearch,Grade_PaidBrand,Grade_PaidNonBrand,Grade_Shopping,Grade_Text,Grade_Social,Grade_Catalog,Grade_Catavg,Date
0,6709175,0.0324,62.2019,0.8909,0.1496,0.0369,0.1127,0.0899,0.0058,0.0983,0.2077,0.0389,0.0265,2.0152,0.2199,F,D,F,F,F,F,F,F,C,F,F,F,2024-04-30
1,5579623,0.0304,59.3459,0.9298,0.4181,0.0347,0.0777,0.1268,0.0034,0.1435,0.5362,0.0190,0.0553,1.8016,0.1265,F,A,F,F,C,F,C,A,F,B,F,F,2024-04-30
2,2812261,0.0366,62.1464,0.9191,0.1397,0.0527,0.3167,0.1045,0.0075,0.1047,0.2258,0.0419,0.0305,2.2763,0.1842,F,D,C,C,F,F,F,F,C,F,F,F,2024-04-30
3,1987891,0.0484,60.2139,0.9117,0.2032,0.0457,0.1234,0.1037,0.0088,0.1713,0.3068,0.0255,0.0334,2.9160,0.2574,F,B,D,F,F,F,B,D,F,F,F,F,2024-04-30
4,6709181,0.0358,62.0899,0.8977,0.1826,0.0376,0.0958,0.0958,0.0071,0.1696,0.2719,0.0357,0.0265,2.2223,0.2273,F,C,F,F,F,F,B,D,C,F,F,F,2024-04-30


In [41]:
df_grades.shape

(7639169, 28)

##### 2.3 Write out df_grades to the local drive (optional)

In [ ]:
# Change the date and write out the file
#df_grades.to_csv('Grades_20240422_V3.csv', index = False)

#### III. Write out df_grades to SQL server

In [7]:
# Create connection to CDR
conn_str1 = f'DRIVER={mssqlDriver};SERVER={mssqlServer};DATABASE={mssqlClient}_CDR;TRUSTED_CONNECTION=no;UID={mssqlUser};PWD={mssqlPw};'
# connect to the SQL Server
conx1 = pyodbc.connect(conn_str1)

In [ ]:
# query to clear the table
drop_table_query = "drop table if exists Score_Output"

In [9]:
# Define function to create table:
def manage_table(connection, query):
    cursor = connection.cursor()
    #try:
    cursor.execute(query)
    connection.commit()


In [42]:
# Clear the table from the previous contents
manage_table(conx1, drop_table_query)

In [11]:
# Create creddentials string and sql alchemy engin (Note: writing out to CDR database)
creds = f'mssql+pyodbc://{mssqlUser}:{mssqlPw}@{mssqlServer}/{mssqlClient}_CDR?driver={mssqlDriver}'
#engine = sa.create_engine(creds, fast_executemany = True)
engine = sa.create_engine(creds, fast_executemany = True, pool_pre_ping= True)


In [ ]:
# Write out the output to the SQL server; specify chunksize to allow execution
df_grades.to_sql("Score_Output", engine, schema='dbo', if_exists = 'append',index = False, chunksize = 1000000)

